In [1]:
!pip install selenium pandas webdriver-manager beautifulsoup4


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import time
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.select import Select
from webdriver_manager.chrome import ChromeDriverManager

def scrape_racefacer(track_config_id, filename, track_label):
    # 1. Setup the Browser (Chrome)
    options = webdriver.ChromeOptions()
    # options.add_argument("--headless")  # Uncomment this line if you don't want to see the browser window
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

    url = "https://www.racefacer.com/en/karting-tracks/pakistan/apexautodromepakistan"
    print(f"\n--- Scraping {track_label} (ID: {track_config_id}) to {filename} ---")
    print(f"Opening {url}...")
    driver.get(url)

    # Allow initial load
    time.sleep(3)

    # Select Track Configuration
    print(f"Selecting Track Configuration {track_label}...")
    try:
        track_select_elem = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.NAME, "track_configuration_id"))
        )
        track_select = Select(track_select_elem)
        track_select.select_by_value(track_config_id)
        print(f"✓ Selected {track_label} successfully")
        
        # Wait for page to reload/update
        time.sleep(5)
    except Exception as e:
        print(f"✗ Could not set track configuration: {e}")

    # Select "All time" from the Period dropdown
    print("Selecting 'All time' from Period dropdown...")
    try:
        # Find the select element by name="period"
        period_select = WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.NAME, "period"))
        )
        
        # Use Select to choose "All time" by value
        select = Select(period_select)
        select.select_by_value('all')
        print("✓ Selected 'All time' successfully")
        
        # Wait for page to reload/update with all-time data
        time.sleep(5)
        print("✓ Page updated with all-time data")
        
    except Exception as e:
        print(f"✗ Could not set period to 'All time': {e}")
        print("Proceeding with default period (Year)...")

    # 2. Loop to load all data
    # Using a more specific selector to count only the actual result rows
    row_selector = ".row .position"
    
    while True:
        try:
            # Count current loaded rows
            current_row_count = len(driver.find_elements(By.CSS_SELECTOR, row_selector))
            print(f"Rows loaded so far: {current_row_count}")

            # Find and Click 'Load More' button
            # We wait up to 10 seconds for the button to be clickable
            try:
                load_more_btn = WebDriverWait(driver, 10).until(
                    EC.element_to_be_clickable((By.CSS_SELECTOR, ".load-more-button"))
                )
            except:
                print("No more 'Load more' buttons found or button not clickable.")
                break
            
            # Scroll to button and click (using JS to avoid interception)
            driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", load_more_btn)
            time.sleep(1) # Small pause to let UI settle
            driver.execute_script("arguments[0].click();", load_more_btn)

            # Smart Wait: Wait until the number of rows actually increases
            # Increased timeout to 20 seconds for larger datasets
            try:
                WebDriverWait(driver, 20).until(
                    lambda d: len(d.find_elements(By.CSS_SELECTOR, row_selector)) > current_row_count
                )
                # Small pause after loading new rows
                time.sleep(1)
            except:
                # Double check if count increased after the timeout
                new_count = len(driver.find_elements(By.CSS_SELECTOR, row_selector))
                if new_count <= current_row_count:
                    print("Timed out waiting for new rows and count did not increase. Assuming end of list.")
                    break
            
        except Exception as e:
            print(f"Stopping due to unexpected error during loading: {e}")
            break

    # 3. Parse the fully loaded HTML
    print("Parsing final data...")
    soup = BeautifulSoup(driver.page_source, 'html.parser')
    data = []

    # --- Extract Podium (1st, 2nd, 3rd) ---
    podium = soup.find('div', class_='track_podium')
    if podium:
        classes = ['first', 'second', 'third']
        ranks = [1, 2, 3]
        for cls, rank in zip(classes, ranks):
            item = podium.find('a', class_=cls)
            if item:
                try:
                    name = item.find('div', class_='name').get_text(strip=True)
                    time_val = item.find('div', class_='time').get_text(strip=True)
                    date = item.find('div', class_='date').get_text(strip=True)
                    link = item.get('href')
                    
                    data.append({
                        'Position': rank,
                        'Name': name,
                        'Date': date,
                        'Max km/h': '', # Podium usually doesn't show these stats
                        'Max G': '',
                        'Best Time': time_val,
                        'Profile URL': link,
                        'Kart Type': track_label
                    })
                except AttributeError:
                    continue

    # --- Extract Table Rows (4 onwards) ---
    rows = soup.find_all('div', class_='row')
    for row in rows:
        try:
            pos_div = row.find('div', class_='position')
            if not pos_div: continue
            pos = pos_div.get_text(strip=True)

            name_div = row.find('div', class_='name')
            name = name_div.get_text(strip=True) if name_div else ""

            date_div = row.find('div', class_='date')
            date = date_div.get_text(strip=True) if date_div else ""

            mk = row.find('div', class_='max-km-h')
            max_km = mk.get_text(strip=True) if mk else ""

            mg = row.find('div', class_='max-g')
            max_g = mg.get_text(strip=True) if mg else ""

            # Time is often inside an anchor tag
            time_a = row.find('a', class_='time')
            if time_a:
                time_span = time_a.find('span')
                best_time = time_span.get_text(strip=True) if time_span else time_a.get_text(strip=True)
            else:
                best_time = ""

            name_link = row.find('a', class_='name-date')
            link = name_link.get('href') if name_link else ""

            data.append({
                'Position': pos,
                'Name': name,
                'Date': date,
                'Max km/h': max_km,
                'Max G': max_g,
                'Best Time': best_time,
                'Profile URL': link,
                'Kart Type': track_label
            })
        except AttributeError:
            continue

    # 4. Save to CSV
    if data:
        df = pd.DataFrame(data)
        # Clean duplicates just in case
        df.drop_duplicates(subset=['Position', 'Name', 'Best Time'], inplace=True)
        
        df.to_csv(filename, index=False)
        print(f"Success! Scraped {len(df)} rows. Saved to {filename}")
        print(df.head())
    else:
        print("No data found.")

    driver.quit()

if __name__ == "__main__":
    # Scrape both track configurations
    track_configs = [
        ("1551", "data_apex_track1.csv", "Track 1"),
        ("1707", "data_apex_track2.csv", "Track 2")
    ]
    
    for config_id, filename, label in track_configs:
        scrape_racefacer(config_id, filename, label)


--- Scraping Track 1 (ID: 1551) to data_apex_track1.csv ---
Opening https://www.racefacer.com/en/karting-tracks/pakistan/apexautodromepakistan...
Selecting Track Configuration Track 1...
✓ Selected Track 1 successfully
Selecting 'All time' from Period dropdown...
✓ Selected 'All time' successfully
✓ Page updated with all-time data
Rows loaded so far: 6
Rows loaded so far: 11
Rows loaded so far: 16
Rows loaded so far: 21
Rows loaded so far: 26
Rows loaded so far: 31
Rows loaded so far: 36
Rows loaded so far: 41
Rows loaded so far: 46
Rows loaded so far: 51
Rows loaded so far: 56
Rows loaded so far: 61
Rows loaded so far: 66
Rows loaded so far: 71
Rows loaded so far: 76
Rows loaded so far: 81
Rows loaded so far: 86
Rows loaded so far: 91
Rows loaded so far: 96
Rows loaded so far: 100
Timed out waiting for new rows and count did not increase. Assuming end of list.
Parsing final data...
Success! Scraped 103 rows. Saved to data_apex_track1.csv
  Position            Name        Date Max km/